# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Let's enumerate the record sets, and for each, the available fields and their `@id`s.

In [ ]:
# List all RecordSets and their fields with @id
record_sets = list(dataset.record_sets)
print(f"Number of RecordSets: {len(record_sets)}\n")
for record_set in record_sets:
    print(f"RecordSet: {record_set.name} (@id: {record_set.id})")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, dtype: {field.data_type})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

First, let's select a record set (we'll use its `@id`), and load it as a pandas DataFrame using `mlcroissant`.

In [ ]:
# Choose a RecordSet @id (replace below with an actual @id from the list above if different)
# For illustration, let's choose the first record set.
record_set_ids = [rset.id for rset in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for RecordSet {record_set_id} with shape: {df.shape}")

# Display columns of the first DataFrame
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in RecordSet {main_record_set_id}:\n{dataframes[main_record_set_id].columns.tolist()}")
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section will include operations like removing outliers, transforming data distributions, or grouping data by key attributes.

Use `@id`s for all field references; you can find the appropriate ones in the cell above.

In [ ]:
# Choose a numeric field and a group field based on available columns
df = dataframes[main_record_set_id]
print(f"Fields available: {df.columns.tolist()}")

# Select a numeric field (replace string with actual field @id as needed)
numeric_field = None
possible_numeric_fields = [c for c in df.columns if df[c].dtype in [int, float, 'int64', 'float64', 'uint8', 'uint16', 'uint32']]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    print(f"Using numeric field: {numeric_field}")
else:
    print("No numeric fields found to analyze.")

if numeric_field:
    # Remove NaN rows for chosen field
    filtered_df = df[df[numeric_field].notna()].copy()
    # Set a threshold, use median for demonstration if many positive values
    if (filtered_df[numeric_field]>0).any():
        threshold = filtered_df[numeric_field].median()
    else:
        threshold = filtered_df[numeric_field].mean()
    filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field (z-score)
    mean = filtered_df[numeric_field].mean()
    std = filtered_df[numeric_field].std()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - mean) / std
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt to group by another field (string field)
    possible_group_fields = [c for c in df.columns if c != numeric_field and (df[c].dtype == 'object' or df[c].dtype.name.startswith('str'))]
    if possible_group_fields:
        group_field = possible_group_fields[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped data by '{group_field}':")
        print(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field, and if a group field is available, show the mean per group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(10,4))
        sns.barplot(x=grouped_df[group_field], y=grouped_df[numeric_field])
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(f'Mean {numeric_field}')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion
We have loaded a multi-table dataset using its Croissant schema, explored the record sets and fields dynamically by their `@id`s, filtered and normalized a quantitative field, grouped the data for summary statistics, and visualized feature distributions using standard Python scientific libraries.

For further analysis, you may wish to select additional record sets and fields by their `@id` for downstream processing or modeling.